In [ ]:
#@title Install and import packages and auth Google sheets
%%capture
!pip -q install playwright nest_asyncio
!playwright install
!playwright install-deps
import nest_asyncio
nest_asyncio.apply()
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import datetime
import asyncio
from gspread_dataframe import get_as_dataframe, set_with_dataframe

# Authenticate
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
#@title Sandbox page pull (FIXED - FT detection)

import re
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

URL = "https://www.totalcorner.com/match/schedule/20260301/page:1"


def split_pair(text):
    if not text:
        return (np.nan, np.nan)
    m = re.search(r'(\d+)\s*-\s*(\d+)', str(text))
    if not m:
        return (np.nan, np.nan)
    return (int(m.group(1)), int(m.group(2)))


def clean_text(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return np.nan
    x = re.sub(r"\s+", " ", str(x)).strip()
    return x if x else np.nan


def get_text_or_none(tag):
    if tag is None:
        return np.nan
    return clean_text(tag.get_text(" ", strip=True))


def extract_match_blocks(soup):
    rows = soup.find_all(attrs={"data-match_id": True})
    seen = set()
    blocks = []
    for row in rows:
        match_id = row.get("data-match_id")
        if not match_id or match_id in seen:
            continue
        seen.add(match_id)
        blocks.append(row)
    return blocks


def extract_basic_fields(block):
    league_tag = block.find("td", class_="text-center td_league")
    home_tag = block.find("td", class_="text-right match_home")
    away_tag = block.find("td", class_="text-left match_away")
    handicap_tag = block.find("td", class_="match_handicap")

    league = get_text_or_none(league_tag)
    home = get_text_or_none(home_tag)
    away = get_text_or_none(away_tag)
    handicap = get_text_or_none(handicap_tag)

    return league, home, away, handicap


def extract_time_status(block):
    # Target the specific span element
    status_span = block.find("span", class_="match_status_minutes")
    if status_span:
        text = status_span.get_text(" ", strip=True).strip().upper()
        if text == "FT" or text == "FULL TIME":
            return "Full"
        if text == "HT" or text == "1ST HALF":
            return "HT"
        if text == "LIVE":
            return "Live"
        # Check for minute markers like "67'"
        if re.search(r"^\d+'$", text):
            return "Live"
        if "POSTPONED" in text:
            return "Postponed"
        if "CANCELLED" in text or "CANCELED" in text:
            return "Cancelled"

    # Fallback to full block text
    text = block.get_text(" ", strip=True).lower()
    if "ft" in text or "full" in text:
        return "Full"
    if "ht" in text or "half" in text:
        return "HT"
    if "live" in text:
        return "Live"
    return np.nan


def extract_score_from_related_block(block):
    candidates = [block]
    nxt = block.find_next_sibling()
    limit = 0
    while nxt is not None and limit < 5:
        candidates.append(nxt)
        nxt = nxt.find_next_sibling()
        limit += 1

    patterns = [
        r'class="[^"]*score[^"]*"[^>]*>\s*(\d+)\s*-\s*(\d+)\s*<',
        r'>\s*(\d+)\s*-\s*(\d+)\s*<',
    ]

    for cand in candidates:
        html = str(cand)
        for p in patterns:
            m = re.search(p, html, flags=re.I)
            if m:
                return int(m.group(1)), int(m.group(2))

    return (np.nan, np.nan)


def extract_corners_from_related_block(block):
    candidates = [block]
    nxt = block.find_next_sibling()
    limit = 0
    while nxt is not None and limit < 5:
        candidates.append(nxt)
        nxt = nxt.find_next_sibling()
        limit += 1

    for cand in candidates:
        full_span = cand.find("span", class_="span_match_corner")
        half_span = cand.find("span", class_="span_half_corner")

        if full_span is not None or half_span is not None:
            full_home, full_away = split_pair(get_text_or_none(full_span))

            half_text = get_text_or_none(half_span)
            m = re.search(r'\(?\s*(\d+)\s*-\s*(\d+)\s*\)?', str(half_text))
            if m:
                half_home, half_away = int(m.group(1)), int(m.group(2))
            else:
                half_home, half_away = np.nan, np.nan

            return full_home, full_away, half_home, half_away

    return (np.nan, np.nan, np.nan, np.nan)


def extract_dangerous_attack_from_related_block(block):
    candidates = [block]
    nxt = block.find_next_sibling()
    limit = 0
    while nxt is not None and limit < 5:
        candidates.append(nxt)
        nxt = nxt.find_next_sibling()
        limit += 1

    for cand in candidates:
        total_div = cand.find("div", class_="match_dangerous_attacks_div")
        half_div = cand.find("div", class_="match_dangerous_attacks_half_div")

        if total_div is not None or half_div is not None:
            total_home, total_away = split_pair(get_text_or_none(total_div))
            half_home, half_away = split_pair(get_text_or_none(half_div))
            return total_home, total_away, half_home, half_away

    return (np.nan, np.nan, np.nan, np.nan)


def parse_totalcorner_html_to_match_df(html):
    soup = BeautifulSoup(html, "html.parser")

    blocked_markers = [
        "just a moment",
        "cf-challenge",
        "turnstile",
        "captcha",
        "attention required",
    ]
    if any(m in html.lower() for m in blocked_markers):
        raise Exception("Blocked by Cloudflare/WAF; Playwright loaded a challenge page.")

    match_blocks = extract_match_blocks(soup)
    rows = []

    for block in match_blocks:
        match_id = block.get("data-match_id")
        league, home, away, handicap = extract_basic_fields(block)
        time_status = extract_time_status(block)

        score_home, score_away = extract_score_from_related_block(block)
        corner_home, corner_away, corner_half_home, corner_half_away = extract_corners_from_related_block(block)
        da_home, da_away, da_half_home, da_half_away = extract_dangerous_attack_from_related_block(block)

        rows.append({
            "League": league,
            "Time": time_status,
            "Score": f"{int(score_home) if pd.notna(score_home) else 0} - {int(score_away) if pd.notna(score_away) else 0} (0 - 0)",
            "Corners": f"{int(corner_home) if pd.notna(corner_home) else 0} - {int(corner_away) if pd.notna(corner_away) else 0} ({int(corner_half_home) if pd.notna(corner_half_home) else 0} - {int(corner_half_away) if pd.notna(corner_half_away) else 0})",
            "DangerousAttack": f"{int(da_home) if pd.notna(da_home) else 0} - {int(da_away) if pd.notna(da_away) else 0} ({int(da_half_home) if pd.notna(da_half_home) else 0} - {int(da_half_away) if pd.notna(da_half_away) else 0})",
            "Home": home,
            "Away": away,
            "Handicap": handicap,
            "MatchId": match_id,
        })

    match_df = pd.DataFrame(rows)

    if "Time" in match_df.columns:
        match_df["Time"] = match_df["Time"].astype(str).str.strip()
        match_df = match_df[match_df["Time"].str.lower() == "full"].copy()

    match_df = match_df[
        [
            "League",
            "Time",
            "Score",
            "Corners",
            "DangerousAttack",
            "Home",
            "Away",
            "Handicap",
            "MatchId",
        ]
    ]

    return match_df


async def fetch_html_with_playwright(url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)

        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
            locale="en-US",
            timezone_id="Europe/Dublin",
            viewport={"width": 1365, "height": 900},
        )

        page = await context.new_page()
        await page.goto(url, wait_until="domcontentloaded", timeout=90000)
        await page.wait_for_timeout(8000)

        html = await page.content()
        title = await page.title()

        await browser.close()
        return title, html


# ============== MAIN ==============

try:
    title, html = await fetch_html_with_playwright(URL)

    # Debug: confirm we got real HTML
    if len(html) < 5000 or any(m in html.lower() for m in ["just a moment", "cf-challenge"]):
        raise Exception(f"HTML fetch failed or blocked: {len(html)} chars, title={title}")

    match_df = parse_totalcorner_html_to_match_df(html)

    if match_df.empty:
        print("WARNING: No matches parsed from HTML.")
    else:
        print(f"Parsed {len(match_df)} matches.")

    match_df["DateID"] = URL.rsplit("/", 2)[-2]
    match_df["HomeLeaguePosition"] = match_df["Home"].str.extract(r"\[(\d+)\]")[0]
    match_df["Home"] = match_df["Home"].str.replace(r"\s*\[\d+\]\s*", " ", regex=True).str.strip()
    match_df["AwayLeaguePosition"] = match_df["Away"].str.extract(r"\[(\d+)\]")[0]
    match_df["Away"] = match_df["Away"].str.replace(r"\s*\[\d+\]\s*", " ", regex=True).str.strip()
    match_df = match_df[['DateID','League','MatchId','Home', 'Away','HomeLeaguePosition','AwayLeaguePosition','Handicap','Score', 'Corners', 'DangerousAttack']]

    # Home Cards Extract (fixed for empty splits)
    homenum = match_df["Home"].str.extract(r"^(\d+(?:\s+\d+)*)", expand=False)
    nums = homenum.fillna("").str.split(expand=True)

    if nums.shape[1] == 0:
        nums[0] = ""
        nums[1] = ""
    elif nums.shape[1] == 1:
        nums[1] = ""

    match_df["HomeRedCard"] = nums[0].where(nums[1].notna())
    match_df["HomeYellowCard"] = nums[1].fillna(nums[0])
    match_df["Home"] = match_df["Home"].str.replace(r"^\d+(?:\s+\d+)*\s+", "", regex=True)

    # Away Cards Extract (fixed for empty splits)
    awaynum = match_df["Away"].str.extract(r"(?<!\S)(\d+(?:\s+\d+)*)$", expand=False)
    nums = awaynum.fillna("").str.split(expand=True)

    if nums.shape[1] == 0:
        nums[0] = ""
        nums[1] = ""
    elif nums.shape[1] == 1:
        nums[1] = ""

    match_df["AwayRedCard"] = nums[0].where(nums[1].notna())
    match_df["AwayYellowCard"] = nums[1].fillna(nums[0])

    mask = awaynum.notna()
    match_df.loc[mask, "Away"] = match_df.loc[mask, "Away"].str.replace(r"\s+\d+(?:\s+\d+)*$", "", regex=True)

    display(match_df.head(3))

except Exception as e:
    print(type(e).__name__, str(e))
    import traceback
    traceback.print_exc()

Parsed 359 matches.


,DateID,League,MatchId,Home,Away,HomeLeaguePosition,AwayLeaguePosition,Handicap,Score,Corners,DangerousAttack,HomeRedCard,HomeYellowCard,AwayRedCard,AwayYellowCard
2,20260301,Ecuador LigaPro Serie A,189905931,Emelec,Delfin SC,NaN,NaN,-0.75,1 - 1 (0 - 0),6 - 1 (2 - 0),45 - 17 (24 - 12),1,5,NaN,4
3,20260301,Guatemala Liga Nacional,189906010,CSD Municipal,Marquense,NaN,NaN,-1.75,1 - 0 (0 - 0),9 - 0 (3 - 0),87 - 27 (40 - 6),NaN,3,NaN,2
4,20260301,Peru Liga 1,189906238,Sport Boys,Deportivo Moquegua,NaN,NaN,-1.0,0 - 0 (0 - 0),2 - 6 (2 - 2),44 - 34 (19 - 12),NaN,2,NaN,5


In [ ]:
# @title PagePull Helper function

async def pull_totalcorner_page(date: str, page_number: int, max_retries: int = 3, base_delay: int = 5):
    """
    Fetch and parse a TotalCorner match schedule page, with retry on failure.

    Args:
        date: Date string in YYYYMMDD format, e.g. "20260301".
        page_number: Page number to fetch, e.g. 1.
        max_retries: Maximum number of attempts before giving up.
        base_delay: Base delay in seconds between retries (multiplied by attempt number).

    Returns:
        pandas.DataFrame of parsed matches, or None if all attempts failed.
    """
    URL = f"https://www.totalcorner.com/match/schedule/{date}/page:{page_number}"

    for attempt in range(1, max_retries + 1):
        try:
            title, html = await fetch_html_with_playwright(URL)
            match_df = parse_totalcorner_html_to_match_df(html)

            match_df["DateID"] = URL.rsplit("/", 2)[-2]
            match_df["HomeLeaguePosition"] = match_df["Home"].str.extract(r"\[(\d+)\]")[0]
            match_df["Home"] = match_df["Home"].str.replace(r"\s*\[\d+\]\s*", " ", regex=True).str.strip()
            match_df["AwayLeaguePosition"] = match_df["Away"].str.extract(r"\[(\d+)\]")[0]
            match_df["Away"] = match_df["Away"].str.replace(r"\s*\[\d+\]\s*", " ", regex=True).str.strip()
            match_df = match_df[["DateID", "League", "MatchId", "Home", "Away",
                                  "HomeLeaguePosition", "AwayLeaguePosition", "Handicap",
                                  "Score", "Corners", "DangerousAttack"]]

            # Home cards
            homenum = match_df["Home"].str.extract(r"^(\d+(?:\s+\d+)*)", expand=False)
            nums = homenum.fillna("").str.split(expand=True)
            nums = nums.reindex(columns=[0, 1])
            match_df["HomeRedCard"] = nums[0].where(nums[1].notna())
            match_df["HomeYellowCard"] = nums[1].fillna(nums[0])
            match_df["Home"] = match_df["Home"].str.replace(r"^\d+(?:\s+\d+)*\s+", "", regex=True)

            # Away cards
            awaynum = match_df["Away"].str.extract(r"(?<!\S)(\d+(?:\s+\d+)*)$", expand=False)
            nums = awaynum.fillna("").str.split(expand=True)
            nums = nums.reindex(columns=[0, 1])
            match_df["AwayRedCard"] = nums[0].where(nums[1].notna())
            match_df["AwayYellowCard"] = nums[1].fillna(nums[0])

            mask = awaynum.notna()
            match_df.loc[mask, "Away"] = match_df.loc[mask, "Away"].str.replace(r"\s+\d+(?:\s+\d+)*$", "", regex=True)
            match_df = match_df[~match_df['League'].str.contains('U20|Women|U23|Reserves|Friendlies|U19', na=False)]
            return match_df

        except Exception as e:
            print(f"Attempt {attempt}/{max_retries} failed for {date} page {page_number}: {type(e).__name__} {str(e)}")
            if attempt < max_retries:
                wait = base_delay * attempt
                print(f"Retrying in {wait}s...")
                await asyncio.sleep(wait)
            else:
                print(f"Giving up on {date} page {page_number} after {max_retries} attempts.")
                return None

In [ ]:
# @title ETL Helper function

SHEET_NAME = "Total Corner Sandbox"

def get_most_recent_dateid(gc, sheet_name=SHEET_NAME, manual_dateid=None):
    if manual_dateid is not None:
        return datetime.datetime.strptime(str(manual_dateid), "%Y%m%d").date()

    sh = gc.open(sheet_name)
    ws = sh.sheet1
    existing_df = get_as_dataframe(ws).dropna(how="all")

    if existing_df.empty or "DateID" not in existing_df.columns:
        raise ValueError("Sheet has no DateID column or is empty; provide manual_dateid.")

    most_recent = existing_df["DateID"].dropna().astype(str).str.replace(r"\.0$", "", regex=True).max()
    return datetime.datetime.strptime(most_recent, "%Y%m%d").date()


async def pull_around_dateid(gc, sheet_name=SHEET_NAME, manual_dateid=None,
                              days_before=2, days_after=2, pages_per_date=3,
                              max_retries=3, base_delay=5, save=True):
    ref_date = get_most_recent_dateid(gc, sheet_name, manual_dateid)
    print(f"Reference DateID: {ref_date.strftime('%Y%m%d')}")

    dates = [
        (ref_date + datetime.timedelta(days=offset)).strftime("%Y%m%d")
        for offset in range(-days_before, days_after + 1)
    ]

    all_dfs = []
    failed_pages = []
    for date_str in dates:
        for page in range(1, pages_per_date + 1):
            print(f"Pulling {date_str} page {page}...")
            df = await pull_totalcorner_page(date_str, page, max_retries=max_retries, base_delay=base_delay)
            if df is not None and not df.empty:
                all_dfs.append(df)
            else:
                failed_pages.append((date_str, page))
            await asyncio.sleep(2)

    if failed_pages:
        print(f"Warning: {len(failed_pages)} page(s) failed after retries: {failed_pages}")

    new_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
    if not new_df.empty:
        new_df["MatchId"] = new_df["MatchId"].astype(str)
        new_df = new_df.drop_duplicates(subset=["MatchId"])

    if save:
        sh = gc.open(sheet_name)
        ws = sh.sheet1
        existing_df = get_as_dataframe(ws).dropna(how="all")

        if not existing_df.empty and "MatchId" in existing_df.columns:
            existing_df["MatchId"] = existing_df["MatchId"].astype(str)
            existing_ids = set(existing_df["MatchId"])
            new_df = new_df[~new_df["MatchId"].isin(existing_ids)]

        if new_df.empty:
            print("No new MatchIds to add.")
            return existing_df

        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
        combined_df = combined_df.drop_duplicates(subset=["MatchId"])

        ws.clear()
        set_with_dataframe(ws, combined_df, include_index=False, resize=True)
        print(f"Added {new_df.shape[0]} new rows. Total rows: {combined_df.shape[0]}")
        return combined_df

    return new_df

In [ ]:
# @title Delete leagues from Gsheet unmask for one time use
# from gspread_dataframe import set_with_dataframe

# sh = gc.open("Total Corner Sandbox")
# ws = sh.sheet1

# # Read and filter
# existing_df = get_as_dataframe(ws).dropna(how="all")
# mask = existing_df['League'].str.contains(
#     'U20|Women|U23|Reserves|Friendlies|U19|U21|U22|Thailand Division 3|Spain Youth League|Poland IV Liga|Play-Offs|Friendly|墨西哥乙級聯賽|France National 3|Guatemala| mins|Esoccer|University|Youth|Shield',
#     na=False
# )
# clean_df = existing_df[~mask].reset_index(drop=True)

# # Clear the sheet and write back only the clean data
# ws.clear()
# set_with_dataframe(ws, clean_df)

In [ ]:
sh = gc.open("Total Corner Sandbox")
ws = sh.sheet1
existing_df = get_as_dataframe(ws).dropna(how="all")
existing_df = existing_df[~existing_df['League'].str.contains('U20|Women|U23|Reserves|Friendlies|U19|Thailand Division 3|Spain Youth League', na=False)]
min_dateid = existing_df["DateID"].dropna().astype(str).str.replace(r"\.0$", "", regex=True).min()
max_dateid = existing_df["DateID"].dropna().astype(str).str.replace(r"\.0$", "", regex=True).max()

max_date = datetime.datetime.strptime(max_dateid, "%Y%m%d").date()
yesterday = datetime.date.today() - datetime.timedelta(days=1)
days_gap = (yesterday - max_date).days #Days from max date till yesterday

# result_df = await pull_around_dateid(gc, manual_dateid=min_dateid, max_retries=3,days_before=10, days_after=0,base_delay=3) #Pull days before df
result_df = await pull_around_dateid(gc, manual_dateid=max_dateid, max_retries=3,days_before=0, days_after=days_gap,base_delay=3) #Pull days after df
result_df.tail(5)

Reference DateID: 20261004
Pulling 20261004 page 1...
Pulling 20261004 page 2...
Pulling 20261004 page 3...
Attempt 1/3 failed for 20261004 page 3: Exception Blocked by Cloudflare/WAF; Playwright loaded a challenge page.
Retrying in 3s...
Attempt 2/3 failed for 20261004 page 3: Exception Blocked by Cloudflare/WAF; Playwright loaded a challenge page.
Retrying in 6s...
Attempt 3/3 failed for 20261004 page 3: Exception Blocked by Cloudflare/WAF; Playwright loaded a challenge page.
Giving up on 20261004 page 3 after 3 attempts.
Pulling 20261005 page 1...
Pulling 20261005 page 2...
Attempt 1/3 failed for 20261005 page 2: Exception Blocked by Cloudflare/WAF; Playwright loaded a challenge page.
Retrying in 3s...
Attempt 2/3 failed for 20261005 page 2: Exception Blocked by Cloudflare/WAF; Playwright loaded a challenge page.
Retrying in 6s...
Attempt 3/3 failed for 20261005 page 2: Exception Blocked by Cloudflare/WAF; Playwright loaded a challenge page.
Giving up on 20261005 page 2 after 3 atte

,DateID,League,MatchId,Home,Away,HomeLeaguePosition,AwayLeaguePosition,Handicap,Score,Corners,DangerousAttack,HomeRedCard,HomeYellowCard,AwayRedCard,AwayYellowCard
159396,20261013,Mexico Copa Promesas,202755511,Titanes de Querétaro,Estudiantes de Querétaro FC,NaN,NaN,-1.0,1 - 2 (0 - 0),6 - 7 (2 - 4),62 - 67 (24 - 33),1,3,NaN,3
159397,20261013,Mexico Copa Promesas,202755514,Queretaro U21,Club Celaya,NaN,NaN,-0.5,2 - 0 (0 - 0),6 - 0 (3 - 0),80 - 26 (44 - 9),NaN,None,NaN,1
159398,20261013,Chile Tercera,202810618,CD Imperial Unido,Deportes Rancagua,NaN,NaN,+0.5,3 - 4 (0 - 0),5 - 4 (4 - 2),59 - 64 (25 - 35),1,3,NaN,3
159399,20261013,Brazil Serie B,202029447,Athletic Club MG,Londrina,15,18,-0.5,2 - 2 (0 - 0),6 - 1 (3 - 0),57 - 22 (19 - 9),NaN,2,NaN,1
159400,20261013,Uruguay Primera Division Amateur,202794505,Deportivo Colonia,IA Sud America,NaN,NaN,-0.75,2 - 2 (0 - 0),8 - 1 (4 - 1),56 - 34 (23 - 20),1,4,4,2
